# 📖 Lab 1: Token Bucket Algorithm

Imagine you're running a popular website. Suddenly, one user starts sending **thousands of requests per second** — maybe they're a bot, or maybe their code has a bug. Without any protection, this could crash your server and ruin the experience for everyone else.

This is where **rate limiting** comes in. Rate limiting controls how many requests a user can make in a given time period. It's like a bouncer at a club — they let people in at a controlled pace, even if there's a huge crowd outside.

The **Token Bucket** algorithm is one of the most popular ways to implement rate limiting. Here's the simple idea:

> Think of a bucket that slowly fills with tokens. Every time someone makes a request, they need to take a token from the bucket. If the bucket is empty, the request is **rejected**. Over time, new tokens drip into the bucket at a steady rate — so the bucket slowly refills.

This algorithm is used by **Stripe**, **AWS**, **GitHub**, and many other major APIs. Let's learn how it works by building one from scratch!

## Learning Objectives

By the end of this notebook, you'll understand:

- 🎯 How the token bucket algorithm works (the core idea)
- 🛠️ How to implement a token bucket from scratch in Python
- 📊 How burst handling works vs. steady-state rate
- ⚖️ How to compare different bucket configurations

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

This starts:
- **Redis** on `localhost:6381` — we'll use this in later notebooks for distributed rate limiting

### Kernel Selection

Make sure you select the **`.venv`** kernel in VS Code:

1. Open this notebook in VS Code
2. Click the kernel picker in the **top-right corner** of the notebook
3. Select the **`.venv`** kernel (Python environment for this lab)
4. If the kernel doesn't appear, reload the VS Code window: `Cmd+Shift+P` → "Reload Window"

> 💡 **Note:** This first notebook only uses Python's built-in `time` module — no extra dependencies needed! Redis will be used in later notebooks.

## 🪣 The Bucket Analogy

Let's understand the token bucket with a simple water bucket analogy:

```
    💧 💧 💧          Tokens drip in at a steady rate
    ↓  ↓  ↓           (refill_rate = tokens per second)
  ┌──────────┐
  │ ░░░░░░░░ │ ← Overflow! Extra tokens are lost
  │ ████████ │         (tokens never exceed max_tokens)
  │ ████████ │
  │ ████████ │ ← Current tokens
  │ ████████ │
  └────┬─────┘
       ↓
  Each request takes
  one token from the bucket
```

Here's how it works:

| Concept | Analogy | In Code |
|---------|---------|--------|
| **Bucket size** | How big the bucket is | `max_tokens` |
| **Water drip rate** | How fast water fills the bucket | `refill_rate` (tokens/sec) |
| **Taking water** | Each request removes one unit | `allow_request()` |
| **Empty bucket** | No water left → request denied | Returns `False` |
| **Overflow** | Bucket is full, extra water is wasted | Tokens capped at `max_tokens` |

### Key Insight

The bucket starts **full**. This means a user can make a **burst** of requests right away (up to `max_tokens`). After the burst, they have to wait for the bucket to refill — which happens at the `refill_rate`. This is why the token bucket is great: it naturally allows short bursts while enforcing a long-term rate.

In [ ]:
import time


class TokenBucket:
    """A simple token bucket rate limiter.

    Think of it as a bucket that slowly fills with tokens:
    - The bucket can hold at most `max_tokens` tokens
    - Tokens are added at `refill_rate` tokens per second
    - Each request costs 1 token
    - If the bucket is empty, the request is denied
    """

    def __init__(self, max_tokens: int, refill_rate: float):
        self.max_tokens = max_tokens
        self.refill_rate = refill_rate  # tokens per second
        self.tokens = max_tokens  # start with a full bucket
        self.last_refill = time.time()

    def _refill(self):
        """Add tokens based on time elapsed since last refill.

        Instead of actually adding tokens every second with a timer,
        we calculate how many tokens SHOULD have been added since
        the last time we checked. This is called 'lazy refill' —
        we only do the math when someone asks for a token.
        """
        now = time.time()
        elapsed = now - self.last_refill
        # How many tokens dripped in during the elapsed time?
        new_tokens = elapsed * self.refill_rate
        # Add new tokens but never exceed the bucket capacity
        self.tokens = min(self.max_tokens, self.tokens + new_tokens)
        self.last_refill = now

    def allow_request(self) -> bool:
        """Check if a request is allowed.

        Returns True if allowed (takes a token), False if denied (bucket empty).
        """
        self._refill()
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

    def get_status(self) -> dict:
        """Get current bucket state (for debugging/learning)."""
        self._refill()
        return {
            "tokens": round(self.tokens, 2),
            "max_tokens": self.max_tokens,
            "refill_rate": self.refill_rate,
        }


# ── Let's try it! ──
# Create a bucket: max 5 tokens, refills at 1 token/second
bucket = TokenBucket(max_tokens=5, refill_rate=1)
print(f"🪣 Initial state: {bucket.get_status()}")
print()
print("The bucket starts full with 5 tokens.")
print("It will refill at 1 token per second.")

## 🚀 Let's See It in Action

Now let's simulate real usage! We'll:

1. **Send a burst of 7 requests instantly** — the bucket only has 5 tokens, so 2 should be denied
2. **Wait 3 seconds** — the bucket should refill with 3 new tokens
3. **Send 2 more requests** — these should be allowed because we waited

Watch how the token count changes with each request:

In [ ]:
import time

# Fresh bucket: 5 tokens max, refills at 1 token/second
bucket = TokenBucket(max_tokens=5, refill_rate=1)

print("=== 💥 Burst: Send 7 requests instantly ===")
print(f"Starting tokens: {bucket.get_status()['tokens']}")
print()

for i in range(7):
    allowed = bucket.allow_request()
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    tokens_left = bucket.get_status()["tokens"]
    print(f"  Request {i+1}: {status}  (tokens left: {tokens_left:.1f})")

print()
print("=== ⏳ Wait 3 seconds for tokens to refill ===")
time.sleep(3)
print(f"After waiting 3s: {bucket.get_status()['tokens']:.1f} tokens")
print()

print("=== 🔄 Send 2 more requests ===")
for i in range(2):
    allowed = bucket.allow_request()
    status = "✅ ALLOWED" if allowed else "❌ DENIED"
    tokens_left = bucket.get_status()["tokens"]
    print(f"  Request {i+1}: {status}  (tokens left: {tokens_left:.1f})")

## ⚖️ Burst vs. Steady Rate

The token bucket has **two key parameters** that control its behavior:

### `max_tokens` — Burst Capacity

This is like the **size of the bucket**. It controls how many requests can happen **all at once**.

- `max_tokens=5` → A user can send 5 requests instantly, then must wait
- `max_tokens=100` → A user can send 100 requests instantly (a big burst!)

### `refill_rate` — Sustained Rate

This is like **how fast water drips in**. It controls the long-term average rate.

- `refill_rate=1` → Over time, the user can sustain 1 request per second
- `refill_rate=10` → Over time, the user can sustain 10 requests per second

### Putting It Together

A bucket with `max_tokens=100` and `refill_rate=10` means:

> "Allow bursts of up to **100 requests**, but sustain only **10 requests/second** over time."

This is perfect for real APIs:
- A web page might need to load 20 resources at once (burst) ✅
- But a user shouldn't be able to make 1000 requests per second forever (sustained limit) 🚫

Let's compare different configurations to see this in action:

In [ ]:
import time


def simulate_bucket(max_tokens, refill_rate, requests_pattern, label):
    """Simulate a token bucket and record what happens.

    Args:
        max_tokens: Maximum tokens the bucket can hold
        refill_rate: How many tokens are added per second
        requests_pattern: list of (delay_seconds, num_requests) tuples
            - delay_seconds: how long to wait before sending requests
            - num_requests: how many requests to send
        label: A friendly name for this configuration
    """
    bucket = TokenBucket(max_tokens=max_tokens, refill_rate=refill_rate)
    results = []
    elapsed = 0

    for delay, count in requests_pattern:
        if delay > 0:
            time.sleep(delay)
            elapsed += delay
        for _ in range(count):
            allowed = bucket.allow_request()
            results.append(
                {
                    "time": round(elapsed, 1),
                    "allowed": allowed,
                    "tokens": round(bucket.tokens, 1),
                }
            )

    allowed_count = sum(1 for r in results if r["allowed"])
    denied_count = sum(1 for r in results if not r["allowed"])
    print(f"\n{label}")
    print(f"  Config: max_tokens={max_tokens}, refill_rate={refill_rate}/sec")
    print(f"  Results: ✅ {allowed_count} allowed, ❌ {denied_count} denied out of {len(results)} total")

    return results


# ── Define a request pattern ──
# Send 10 requests immediately, wait 2 seconds, then send 5 more
pattern = [(0, 10), (2, 5)]

print("Comparing different bucket configurations:")
print("Pattern: 10 requests → wait 2s → 5 requests")

# Small bucket, slow refill
simulate_bucket(3, 1, pattern, "🪣 Small bucket (3 tokens, 1/sec refill)")

# Medium bucket, medium refill
simulate_bucket(10, 2, pattern, "🪣 Medium bucket (10 tokens, 2/sec refill)")

# Large bucket, fast refill
simulate_bucket(20, 5, pattern, "🪣 Large bucket (20 tokens, 5/sec refill)")

## 🔮 Token Bucket vs. Other Algorithms — Preview

The token bucket is just **one** way to do rate limiting. In the next notebooks, we'll explore other approaches:

| Algorithm | How It Works | Best For |
|-----------|-------------|----------|
| **Token Bucket** (this notebook!) | Bucket fills with tokens over time | Allowing bursts + sustained rate |
| **Sliding Window Counter** (next notebook) | Counts requests in a rolling time window | Smooth, predictable limits |

### Why is Token Bucket so popular?

- ✅ **Simple to implement** — just a few lines of code (you just did it!)
- ✅ **Handles bursts naturally** — the bucket starts full, so short bursts are fine
- ✅ **Low memory** — only needs 2 numbers per client (token count + last refill time)
- ✅ **Battle-tested** — used by Stripe, AWS, GitHub, and many production systems

### Who uses it?

- **Stripe** uses token bucket for their API rate limits
- **AWS API Gateway** uses token bucket for throttling
- **NGINX** uses a variation called "leaky bucket" for request limiting

## 📝 Key Takeaways

Let's recap what we learned:

1. **Token bucket controls both burst and sustained rate** — the bucket size sets the burst limit, and the refill rate sets the sustained limit

2. **Two parameters are all you need:**
   - `max_tokens` = burst size (how many requests at once)
   - `refill_rate` = sustained rate (how many requests per second over time)

3. **Implementation is simple** — just track the current token count and the last refill time. Use "lazy refill" to calculate tokens only when needed.

4. **Each request costs 1 token** — if the bucket is empty, the request is denied. No tokens, no service!

5. **Used in production everywhere** — Stripe, AWS, GitHub, and many other major APIs rely on this algorithm

6. **Memory efficient** — only 2 values per client (`tokens` + `last_refill`). Even with millions of users, this is tiny!

## ➡️ What's Next?

In the next notebook, we'll explore the **Sliding Window Counter** algorithm — a different approach to rate limiting that provides smoother, more predictable limits.

While the token bucket allows bursts (which is great for many use cases), the sliding window counter spreads the limit evenly across time. We'll implement it, compare it to the token bucket, and discuss when to use which approach.

See you in **Lab 2**! 🚀